# AML Signal Escalation — Submission Pipeline

Track A (feature engineering) va Track B (modeling) qismlarini boshidan oxirigacha birlashtiruvchi reproducible notebook.
Ishga tushirishdan oldin loyiha ildizidan `pip install -r requirements.txt` bajaring.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

TEAM_ID = "PLACEHOLDER"  # TODO: haqiqiy team ID bilan almashtiring

from src import config
from src.data_loading import load_signals, load_transactions
from src.features import build
from src.model import train, cross_validate
from src.predict import predict, validate_submission, write

## 1. Load raw data

In [2]:
train_signals = load_signals(config.TRAIN_SIGNALS_PATH)
train_transactions = load_transactions(config.TRAIN_TRANSACTIONS_PATH)
test_signals = load_signals(config.TEST_SIGNALS_PATH)
test_transactions = load_transactions(config.TEST_TRANSACTIONS_PATH)
train_signals.shape, train_transactions.shape, test_signals.shape, test_transactions.shape

((14000, 3), (6987663, 5), (6000, 2), (3027575, 5))

## 2. Build features (Track A)

In [3]:
train_features = build(train_signals, train_transactions)
test_features = build(test_signals, test_transactions)
train_features.head()

,signal_id,n_txn,amt_mean,amt_std,amt_max,amt_sum,frac_kirim,frac_karta,frac_bank_otkazmasi,frac_naqd,...,n_txn_1d,n_txn_7d,n_txn_30d,span_days,velocity,hour_entropy,hour_maxshare,dow_entropy,dow_maxshare,eskalatsiya
0,SG_000002,203,0.437551,1.283484,3.816146,88.822846,0.665025,0.379310,0.546798,0.073892,...,25.0,33.0,76,116.957928,1.720953,3.041451,0.128079,1.901236,0.246305,0
1,SG_000003,271,0.327211,0.714104,3.806999,88.674149,0.675277,0.254613,0.642066,0.095941,...,18.0,23.0,39,178.943530,1.506028,3.102211,0.103321,1.916735,0.225092,0
2,SG_000004,760,0.025983,0.997670,4.498756,19.747271,0.828947,0.622368,0.357895,0.019737,...,77.0,86.0,150,179.814097,4.203212,3.097765,0.134211,1.914601,0.226316,0
3,SG_000005,432,0.278599,0.852941,3.418567,120.354553,0.701389,0.546296,0.400463,0.046296,...,51.0,55.0,79,179.754815,2.389978,3.050518,0.159722,1.899515,0.252315,0
4,SG_000006,392,0.643715,0.973805,3.578128,252.336357,0.668367,0.397959,0.515306,0.084184,...,58.0,64.0,99,176.843912,2.204180,3.027155,0.178571,1.882093,0.278061,0


## 3. Train + cross-validate (Track B)

In [4]:
X = train_features[config.FEATURE_COLUMNS]
y = train_signals[config.TARGET_COL]

auc = cross_validate(X, y)
print(f"CV ROC-AUC: {auc:.4f}")

model = train(X, y)

CV ROC-AUC: 0.5649


## 4. Predict + write submission

In [5]:
predictions = predict(model, test_features)
validate_submission(predictions, test_signals[config.ID_COL])
out_path = str(config.OUTPUTS_DIR / f"team_{TEAM_ID}.csv")
write(predictions, out_path)
print(f"wrote {out_path}")
predictions.head()

wrote /home/jayxun/Desktop/Hackathon/WUIT Hackathon/outputs/team_PLACEHOLDER.csv


,signal_id,ehtimollik
0,SG_000001,0.167574
1,SG_000007,0.191159
2,SG_000009,0.135877
3,SG_000010,0.127799
4,SG_000011,0.189088
